# Generate Paper Outputs — Full Scale

Evaluates the trained MFFT checkpoints (tiny/base/large) on the **shared
stratified test split** and aggregates every result produced by the other
notebooks (baselines, ablations, LOGO, paper evals) into manuscript-ready
tables and figures.

Supersedes the legacy `generate_paper_outputs.py`, which used the old
`paper/results/` path and built its own random split instead of the shared
`split_indices*.json` protocol.

**Run this last**, after the training notebooks have produced their
checkpoints and result files.


In [ ]:
# Cell 1: Imports & Environment Setup (Kaggle/DGX/Local auto-detect)
import os, sys, math, json, time, random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from tqdm.notebook import tqdm

# ── Kaggle / DGX / Local auto-detection ──
sys.path.insert(0, str(Path.cwd().resolve() / 'model'))
from src.kaggle_utils import KaggleEnv
env = KaggleEnv(project_root_search=True)
PROJECT_ROOT = env.project_root
os.chdir(env.working_dir)

# ── Reproducibility ──
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SMOKE_TEST = not torch.cuda.is_available()
print(f'Device: {device} | SMOKE_TEST: {SMOKE_TEST}')

In [ ]:
# Cell 2: Config, shared split, paths
from src.dataset import create_split_dataloaders
from src.model import build_mfft, count_parameters

IMAGE_SIZE = 224 if (SMOKE_TEST or QUICK_5K) else 384
BATCH = 32 if QUICK_5K else (8 if SMOKE_TEST else 64)
MAX_SAMPLES = 5000 if QUICK_5K else (600 if SMOKE_TEST else None)

# ── Manifest: try local, then download from HF ──
_manifest = PROJECT_ROOT / 'dataset' / 'metadata' / 'train_manifest.csv'
if not _manifest.exists() or _manifest.stat().st_size < 1000:
    env.download_manifest(_manifest)
_quick5k_manifest = PROJECT_ROOT / 'dataset' / 'metadata' / 'test_train_manifest.csv'
if QUICK_5K and _quick5k_manifest.exists():
    _manifest = _quick5k_manifest
    print(f'QUICK_5K: using prepared manifest {_manifest.name}')
elif not _manifest.exists():
    _manifest = PROJECT_ROOT / 'dataset' / 'metadata' / 'clean_metadata.csv'
    print('WARNING: using clean_metadata.csv (run prepare_training_manifest.py for full scale)')
MANIFEST = str(_manifest)

_split_name = ('split_indices_quick5k.json' if QUICK_5K
               else 'split_indices_smoke.json' if SMOKE_TEST
               else 'split_indices.json')
_, _, test_loader = create_split_dataloaders(
    root_dir=str(PROJECT_ROOT), metadata_paths=[MANIFEST],
    batch_size=BATCH, num_workers=0 if (SMOKE_TEST or QUICK_5K) else 8,
    size=IMAGE_SIZE, val_split=0.10, test_split=0.10, seed=SEED,
    use_weighted_sampler=False,
    split_index_path=str(PROJECT_ROOT / 'dataset' / 'metadata' / _split_name),
    max_samples=MAX_SAMPLES,
)
test_dataset = test_loader.dataset

CKPT_ROOT = PROJECT_ROOT / 'model' / 'checkpoints' / ('verify' if MODE == 'verify' else '')
RESULTS_ROOT = PROJECT_ROOT / 'paper' / 'result' / MODE
OUT = RESULTS_ROOT / 'paper_outputs'
OUT.mkdir(parents=True, exist_ok=True)
print(f'Test samples: {len(test_dataset)}')
print(f'Checkpoints:  {CKPT_ROOT}')
print(f'Outputs   ->  {OUT}')


In [ ]:
# Cell 3: Evaluate MFFT tiny/base/large on the shared test split
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score

def find_checkpoint(variant):
    d = CKPT_ROOT / f'{variant}_model'
    for name in ('best.pt', f'best_mfft_{variant}.pt',
                 f'mfft_{variant}_final.pt', 'final.pt'):
        if (d / name).exists():
            return d / name
    return None

def evaluate_loader(model):
    model.eval()
    ys, ps = [], []
    with torch.no_grad():
        for images, labels in tqdm(test_loader, desc='eval', leave=False):
            probs = F.softmax(model(images.to(device)), dim=-1)[:, 1]
            ys.extend(labels.numpy().tolist())
            ps.extend(probs.cpu().numpy().tolist())
    y_true, y_score = np.array(ys), np.array(ps)
    y_pred = (y_score >= 0.5).astype(int)
    return y_true, y_score, y_pred

mfft_rows, mfft_evals = [], {}
for variant in ('tiny', 'base', 'large'):
    ckpt = find_checkpoint(variant)
    if ckpt is None:
        print(f'MFFT-{variant}: no checkpoint under {CKPT_ROOT / (variant + "_model")} — skipped')
        continue
    model = build_mfft(variant)
    state = torch.load(ckpt, map_location=device, weights_only=True)
    model.load_state_dict(state.get('model_state_dict', state))
    model = model.to(device)
    print(f'MFFT-{variant}: {count_parameters(model):,} params ({ckpt.relative_to(PROJECT_ROOT)})')

    y_true, y_score, y_pred = evaluate_loader(model)
    cm = np.zeros((2, 2), dtype=int)
    for t, p in zip(y_true, y_pred):
        cm[t, p] += 1
    metrics = {
        'model': f'MFFT-{variant}',
        'params': count_parameters(model),
        'accuracy': round(float((y_pred == y_true).mean() * 100), 2),
        'precision': round(float(precision_score(y_true, y_pred, zero_division=0) * 100), 2),
        'recall': round(float(recall_score(y_true, y_pred, zero_division=0) * 100), 2),
        'f1': round(float(f1_score(y_true, y_pred, zero_division=0) * 100), 2),
        'auc_roc': round(float(roc_auc_score(y_true, y_score)), 4),
        'checkpoint': str(ckpt.relative_to(PROJECT_ROOT)),
    }
    vdir = OUT / f'{variant}_model'
    vdir.mkdir(parents=True, exist_ok=True)
    (vdir / 'metrics.json').write_text(json.dumps(metrics, indent=2))
    (vdir / 'confusion_matrix.json').write_text(json.dumps(
        {'tn': int(cm[0, 0]), 'fp': int(cm[0, 1]),
         'fn': int(cm[1, 0]), 'tp': int(cm[1, 1])}, indent=2))
    mfft_rows.append(metrics)
    mfft_evals[variant] = {'y_true': y_true, 'y_score': y_score, 'cm': cm, 'model': model}
    print(f"  acc={metrics['accuracy']}% f1={metrics['f1']}% auc={metrics['auc_roc']}")
    del model

if mfft_rows:
    mfft_df = pd.DataFrame(mfft_rows).set_index('model')
    mfft_df.to_csv(OUT / 'mfft_summary.csv')
    print(f'\nSaved {OUT / "mfft_summary.csv"}')
    print(mfft_df.to_string())
else:
    print('No MFFT checkpoints found — train the MFFT notebooks first.')


In [ ]:
# Cell 4: Manuscript figures (for the best available variant)
from src.visualize import generate_all_figures

FIG_VARIANT = 'base' if 'base' in mfft_evals else (next(iter(mfft_evals), None))
if FIG_VARIANT is None:
    print('No evaluated MFFT model — skipping figures.')
else:
    ev = mfft_evals[FIG_VARIANT]
    figs_dir = OUT / 'fig'
    figs_dir.mkdir(parents=True, exist_ok=True)
    sample_img = test_dataset.samples[0][0] if len(test_dataset.samples) else None
    print(f'Figures for MFFT-{FIG_VARIANT} -> {figs_dir}')
    generate_all_figures(
        model=ev['model'].to(device),
        val_loader=test_loader,
        device=device,
        y_true=ev['y_true'],
        y_score=ev['y_score'],
        cm=ev['cm'],
        labels=[s[1] for s in test_dataset.samples],
        sample_image_path=sample_img,
        output_dir=figs_dir,
    )


In [ ]:
# Cell 5: Aggregate every result under RESULTS_ROOT into master tables
rows = []
for mf in sorted(RESULTS_ROOT.rglob('metrics.json')):
    try:
        data = json.loads(mf.read_text())
    except Exception as e:
        print(f'  skip {mf}: {e}')
        continue
    data['_source'] = str(mf.parent.relative_to(RESULTS_ROOT))
    rows.append(data)

if rows:
    all_df = pd.DataFrame(rows)
    front = [c for c in ('model', 'accuracy', 'test_accuracy', 'precision',
                         'test_precision', 'recall', 'test_recall', 'f1',
                         'test_f1', 'auc_roc', 'test_auc', 'params',
                         '_source') if c in all_df.columns]
    all_df = all_df[front + [c for c in all_df.columns if c not in front]]
    all_df.to_csv(OUT / 'all_models_summary.csv', index=False)
    print(f'Saved {OUT / "all_models_summary.csv"} ({len(all_df)} models)')
    print(all_df[front].to_string(index=False))
else:
    print(f'No metrics.json files found under {RESULTS_ROOT}')

print('\n=== Other artifacts found ===')
for pattern in ('baselines_model/baseline_summary.csv', 'ablation/ablation_summary.csv',
                'logo/*.csv', 'paper_evals/*'):
    for p in sorted(RESULTS_ROOT.glob(pattern)):
        print(f'  {p.relative_to(PROJECT_ROOT)}')


In [ ]:
# Cell: Upload paper outputs to HuggingFace
print('\nUploading paper outputs to HuggingFace...')
mode = 'verify' if SMOKE_TEST else 'full_scale'
paper_outputs_dir = PROJECT_ROOT / 'paper' / 'result' / mode / 'paper_outputs'

if paper_outputs_dir.exists():
    # Upload all files in paper_outputs root (CSVs)
    for f in paper_outputs_dir.iterdir():
        if f.is_file() and f.suffix in ('.json', '.csv'):
            env.upload_to_hf(f, env.hf_results_repo, f'results/paper_outputs/{f.name}')
    
    # Upload fig/ subdirectory
    fig_dir = paper_outputs_dir / 'fig'
    if fig_dir.exists():
        for f in fig_dir.iterdir():
            if f.is_file() and f.suffix == '.png':
                env.upload_to_hf(f, env.hf_results_repo, f'results/paper_outputs/fig/{f.name}')

print('Paper outputs upload complete')

## Outputs

- `paper_outputs/mfft_summary.csv` — MFFT tiny/base/large on the shared test split
- `paper_outputs/<variant>_model/metrics.json` + `confusion_matrix.json`
- `paper_outputs/fig/` — manuscript figures (frequency decomposition, ROC, calibration, ...)
- `paper_outputs/all_models_summary.csv` — every model found under this mode's result tree

Baseline/ablation/LOGO/paper-eval tables are produced by their own notebooks;
this notebook locates and lists them so nothing is missed when writing the paper.
